In [ ]:
# 준비 코드

# 1. 필수 패키지 설치
!pip install factor_analyzer

# 2. 라이브러리 불러오기
import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
from sklearn.decomposition import PCA
import platform
import matplotlib.font_manager as fm

# 3. 한글 폰트 설정 (운영체제별)
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')

elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')

else:
    # GitHub에서 폰트 다운로드
    font_url = "https://raw.githubusercontent.com/shimoohyun/data/main/NanumGothic.ttf"
    font_path = "./NanumGothic.ttf"

    if not os.path.exists(font_path):
        urllib.request.urlretrieve(font_url, font_path)

    # 폰트 등록
    fm.fontManager.addfont(font_path)
    font = fm.FontProperties(fname=font_path)

    # 폰트 적용
    plt.rc('font', family=font.get_name())

# 마이너스 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# 4. 데이터 불러오기 및 전처리 (psych::bfi 데이터셋)
url = "https://raw.githubusercontent.com/shimoohyun/data/main/validity.csv"
qul = pd.read_csv(url)
qul_items = qul.loc[:, 'Q3_1':'Q6_5'].dropna()
print(qul_items.head())

In [ ]:
# 5. 요인분석 타당성 검토
# (1) KMO
kmo_all, kmo_model = calculate_kmo(qul_items)
print(f"KMO 전체 지수: {round(kmo_model, 2)}")
# (2) Bartlett의 구형성 검정
chi2, p_value = calculate_bartlett_sphericity(qul_items)
print(f"Bartlett's Chi2 = {round(chi2, 2)}, p-value = {p_value}")

In [ ]:
# 6. 요인 수 결정: Scree Plot (빨간선 제거)
pca = PCA()
pca.fit(qul_items)
ev = pca.explained_variance_
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(ev)+1), ev, marker='o')
plt.title("스크리 플롯 (Scree Plot)")
plt.xlabel("요인 번호")
plt.ylabel("고유값 (Eigenvalue)")
plt.grid(True)
plt.show()
print("고유값 > 1인 요인의 수:", sum(ev > 1))

In [ ]:
# 7. 탐색적 요인분석 (EFA) 및 결과 출력
fa = FactorAnalyzer(n_factors=4, rotation='varimax', method='principal')
fa.fit(qul_items)

# (1) 요인 부하량
loadings = pd.DataFrame(fa.loadings_, index=qul_items.columns, columns=[f"요인{i+1}" for i in range(fa.n_factors)])
print("\n[요인 부하량]\n", loadings.round(2))
# (2) 공통성
communalities = pd.Series(fa.get_communalities(), index=qul_items.columns)
print("\n[공통성]\n", communalities.round(2))
# (3) 고유성
uniqueness = pd.Series(fa.get_uniquenesses(), index=qul_items.columns)
print("\n[고유성]\n", uniqueness.round(2))
# (4) 고유값
eigenvalues, _ = fa.get_eigenvalues()
print("\n[고유값]\n", eigenvalues.round(2))